# ARMAS 5-minute time-series exploration

This notebook is the data-first checkpoint for the project **Temporal Deep Learning for Aviation Effective Dose-Rate Prediction**.

It answers four questions before modeling:

1. Do P1, P2, and P3 have the expected schema and 12-step histories?
2. How do the ARMAS target and the high-radiation regime (`ARMAS > 15 µSv/h`) differ by partition?
3. How strong is the NAIRAS V3 physics benchmark?
4. Are there obvious missingness, non-finite values, duplicates, or distribution shifts?

Modeling contract: P1 is training, P2 is validation, and P3 is untouched final testing in the first modeling pass. `Vehicle_ID`, `NAIRASV2`, and `NAIRASV3` are never predictors. NAIRAS V3 is only a benchmark.

## 0. Setup

Run locally from the repository root, or set `DATA_DIR` to the dataset folder in Google Drive. The loader streams JSON Lines and stores only scalar fields plus the current (last) value of each temporal sequence, keeping exploration memory-friendly.

In [ ]:
# Uncomment only if your environment is missing these packages.
# %pip install -q pandas numpy matplotlib seaborn scikit-learn

import json
import math
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
RNG = np.random.default_rng(42)

In [ ]:
# Local default. In Colab, point this to the folder in Drive instead.
DATA_DIR = Path("ARMASdatasets_COMPLETE-20261006T110557Z-1-001/ARMASdatasets_COMPLETE")
# Example Colab path:
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/ARMASdatasets_COMPLETE')

FILES = {
    p: DATA_DIR / f"ARMAS_TIMESERIES_5M_20132023_{p}.json"
    for p in ("P1", "P2", "P3")
}
assert all(path.exists() for path in FILES.values()), (
    "Set DATA_DIR to the folder containing the three ARMAS_TIMESERIES_5M files."
)
FILES

## 1. Stream and validate the partitions

Prediction: each record should contain 8 static variables, 35 list-valued environmental variables, and every list should have length 12. A cross-check against the matching static file shows that index `0` is the current-time value; later indices step backward into the preceding hour.

In [ ]:
STATIC_FEATURES = [
    "Latitude", "Longitude", "Altitude(Bar)", "Altitude(GPS)",
    "Geomagnetic_latitude", "Geomagnetic_longitude",
    "Geomagnetic_Rc", "Geomagnetic_Lshell",
]
EXCLUDED_PREDICTORS = ["Vehicle_ID", "NAIRASV2", "NAIRASV3"]
META_COLUMNS = ["Datetime", "ARMAS", "NAIRASV3", "NAIRASV2", "Vehicle_ID"]

def stream_partition(path, sequence_sample_size=500):
    compact_rows = []
    sequence_sample = []
    temporal_features = None
    length_sets = defaultdict(set)
    nonfinite = defaultdict(int)
    keys = None

    with path.open("r", encoding="utf-8") as handle:
        for row_index, line in enumerate(handle):
            if not line.strip():
                continue
            row = json.loads(line)
            keys = keys or list(row)
            if temporal_features is None:
                temporal_features = [k for k, v in row.items() if isinstance(v, list)]

            compact = {k: row[k] for k in META_COLUMNS + STATIC_FEATURES}
            for feature in temporal_features:
                values = row[feature]
                length_sets[feature].add(len(values))
                arr = np.asarray(values, dtype=np.float64)
                nonfinite[feature] += int((~np.isfinite(arr)).sum())
                compact[f"{feature}__current"] = values[0] if values else np.nan
            compact_rows.append(compact)

            # Reservoir sample: representative sequences without retaining the full raw dataset.
            if len(sequence_sample) < sequence_sample_size:
                sequence_sample.append({k: row[k] for k in temporal_features})
            else:
                j = int(RNG.integers(0, row_index + 1))
                if j < sequence_sample_size:
                    sequence_sample[j] = {k: row[k] for k in temporal_features}

    frame = pd.DataFrame(compact_rows)
    frame["Datetime"] = pd.to_datetime(frame["Datetime"], utc=True)
    audit = {
        "keys": keys,
        "temporal_features": temporal_features,
        "length_sets": dict(length_sets),
        "nonfinite": dict(nonfinite),
    }
    return frame, sequence_sample, audit

compact, samples, audits = {}, {}, {}
for partition, path in FILES.items():
    compact[partition], samples[partition], audits[partition] = stream_partition(path)
    print(partition, compact[partition].shape)

In [ ]:
TEMPORAL_FEATURES = audits["P1"]["temporal_features"]
assert len(STATIC_FEATURES) == 8
assert len(TEMPORAL_FEATURES) == 35
assert all(audits[p]["temporal_features"] == TEMPORAL_FEATURES for p in FILES)
assert all(lengths == {12} for p in FILES for lengths in audits[p]["length_sets"].values())
assert all(count == 0 for p in FILES for count in audits[p]["nonfinite"].values())
assert all(compact[p].isna().sum().sum() == 0 for p in FILES)
print("PASS: 8 static features, 35 temporal features, 12 steps, no null/non-finite values.")

In [ ]:
# Verify temporal ordering against the independently supplied static dataset.
static_path = DATA_DIR / "ARMAS_STATIC_20132023_P1.json"
if static_path.exists():
    with static_path.open("r", encoding="utf-8") as handle:
        static_first = json.loads(next(handle))
    with FILES["P1"].open("r", encoding="utf-8") as handle:
        sequence_first = json.loads(next(handle))
    assert static_first["Datetime"] == sequence_first["Datetime"]
    assert all(np.isclose(static_first[f], sequence_first[f][0], equal_nan=True)
               for f in TEMPORAL_FEATURES)
    print("PASS: sequence index 0 matches the current-time static values.")
else:
    print("Static file not found; current-time ordering could not be independently rechecked.")

Reflection: if this assertion fails on a future dataset version, stop. Do not silently pad, truncate, or impute until the failure pattern is understood and a train-only preprocessing rule is chosen.

## 2. Partition overview and target distribution

In [ ]:
overview = []
for partition, df in compact.items():
    overview.append({
        "partition": partition,
        "rows": len(df),
        "start": df["Datetime"].min(),
        "end": df["Datetime"].max(),
        "ARMAS_mean": df["ARMAS"].mean(),
        "ARMAS_median": df["ARMAS"].median(),
        "ARMAS_p95": df["ARMAS"].quantile(0.95),
        "ARMAS_max": df["ARMAS"].max(),
        "high_n": (df["ARMAS"] > 15).sum(),
        "high_pct": 100 * (df["ARMAS"] > 15).mean(),
    })
overview = pd.DataFrame(overview).set_index("partition")
overview.round(3)

In [ ]:
full_data = pd.concat(compact.values(), ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(full_data["ARMAS"], bins=60, stat="density", element="step",
             fill=True, alpha=.25, color="#2E5EAA", label="Full dataset", ax=axes[0])
sns.ecdfplot(full_data, x="ARMAS", color="#2E5EAA", label="Full dataset", ax=axes[1])
axes[0].axvline(15, color="black", linestyle="--", label="high-dose threshold")
axes[0].set(xlabel="ARMAS (µSv/h)", title="Target distribution - full dataset")
axes[1].axvline(15, color="black", linestyle="--")
axes[1].set(xlabel="ARMAS (µSv/h)", title="Empirical CDF - full dataset")
axes[0].legend(); axes[1].legend();
plt.tight_layout()

Questions:

- Are the three curves close enough that one partition is not an unusually easy or hard test set?
- Does the high-dose fraction support reporting both overall and `ARMAS > 15` metrics?

## 3. NAIRAS V3 benchmark

This is a benchmark only. It is not included in `X`, and NAIRAS V2 is neither a predictor nor the primary benchmark.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "n": len(y_true),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "bias": float(np.mean(np.asarray(y_pred) - np.asarray(y_true))),
    }

benchmark_rows = []
for partition, df in compact.items():
    for regime, mask in {
        "all": np.ones(len(df), dtype=bool),
        "ARMAS > 15": df["ARMAS"].to_numpy() > 15,
    }.items():
        result = regression_metrics(df.loc[mask, "ARMAS"], df.loc[mask, "NAIRASV3"])
        benchmark_rows.append({"partition": partition, "regime": regime, **result})
benchmark = pd.DataFrame(benchmark_rows)
benchmark.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, (partition, df) in zip(axes, compact.items()):
    sample = df.sample(min(6000, len(df)), random_state=42)
    ax.hexbin(sample["ARMAS"], sample["NAIRASV3"], gridsize=45, mincnt=1, cmap="viridis")
    limits = [0, max(sample["ARMAS"].max(), sample["NAIRASV3"].max())]
    ax.plot(limits, limits, "r--", linewidth=1)
    ax.set(title=partition, xlabel="Measured ARMAS", ylabel="NAIRAS V3")
fig.suptitle("Physics benchmark versus measurements")
plt.tight_layout()

## 4. Static/current feature relationships and shift

Correlation is descriptive, not a feature-selection rule. The deep models will receive all proposal-approved inputs.

In [ ]:
train_df = compact["P1"]
numeric_predictors = STATIC_FEATURES + [f"{f}__current" for f in TEMPORAL_FEATURES]
corr = train_df[numeric_predictors + ["ARMAS"]].corr(method="spearman")["ARMAS"].drop("ARMAS")
corr.reindex(corr.abs().sort_values(ascending=False).index).head(20).to_frame("Spearman_with_ARMAS")

In [ ]:
top = corr.abs().nlargest(15).index
signed = corr.loc[top].sort_values()
plt.figure(figsize=(8, 5))
colors = np.where(signed >= 0, "#2a9d8f", "#e76f51")
plt.barh(signed.index, signed.values, color=colors)
plt.xlabel("Spearman correlation with ARMAS (P1 only)")
plt.title("Strongest monotonic relationships in training data")
plt.tight_layout()

In [ ]:
# Standardized mean difference relative to P1; large magnitudes flag covariate shift.
p1_mean = train_df[numeric_predictors].mean()
p1_std = train_df[numeric_predictors].std().replace(0, np.nan)
shift = pd.DataFrame({
    p: (compact[p][numeric_predictors].mean() - p1_mean) / p1_std
    for p in ("P2", "P3")
})
shift["max_abs_shift"] = shift.abs().max(axis=1)
shift.sort_values("max_abs_shift", ascending=False).head(20).round(3)

## 5. Inspect the one-hour trajectories

Each line below is one 12-step history. Index 0 matches the static current-time dataset; the plot reverses the stored order so time reads naturally from 55 minutes ago to the current observation.

In [ ]:
representative_features = [
    "NM_OULU", "SXR_long", "Particles_P10", "SW_Bz", "Index_Dst", "Solar_f107"
]
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
minutes = np.arange(-55, 5, 5)
for ax, feature in zip(axes.flat, representative_features):
    arr = np.asarray([row[feature][::-1] for row in samples["P1"]], dtype=float)
    q10, q50, q90 = np.quantile(arr, [0.1, 0.5, 0.9], axis=0)
    ax.fill_between(minutes, q10, q90, alpha=0.25)
    ax.plot(minutes, q50, linewidth=2)
    ax.set(title=feature, xlabel="minutes relative to output", ylabel="value")
fig.suptitle("P1 temporal profiles: median and 10–90% interval")
plt.tight_layout()

## 6. Duplicate/leakage audit

The dataset documentation states that flights are separated by partition. Since there is no explicit flight ID, this checks exact observation duplicates using time, position, altitude, and target. It cannot prove flight separation; that guarantee must come from dataset construction.

In [ ]:
duplicate_key = ["Datetime", "Latitude", "Longitude", "Altitude(Bar)", "ARMAS"]
signatures = {
    p: set(pd.util.hash_pandas_object(df[duplicate_key], index=False).astype(str))
    for p, df in compact.items()
}
overlap = pd.DataFrame([
    {"pair": f"{a}-{b}", "exact_overlap": len(signatures[a] & signatures[b])}
    for a, b in (("P1", "P2"), ("P1", "P3"), ("P2", "P3"))
])
overlap

## 7. Exploration decision gate

Before moving to modeling, record answers to these prompts:

- Did all schema assertions pass?
- Are any partition shifts large enough to mention in the report?
- How does NAIRAS V3 performance change above 15 µSv/h?
- Do the temporal profiles look physically plausible, and is index 0 confirmed as current time?
- Are exact cross-partition duplicates zero?

If these checks pass, continue to `02_stepwise_gpu_models_colab.ipynb`.